# Bike Rental Demand Analysis

## Project overview

This case study uses bike-sharing demand data to examine how rental activity changes across time, seasons and weather conditions. The focus is on practical data preparation with pandas, datetime handling, grouping and exploratory visualization.

### Analytical questions

- At what hours is demand highest?
- How does average demand vary by season?
- Is demand different on working and non-working days?
- How are temperature and humidity associated with rental volume?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/bike_sharing_sample.csv")
df = pd.read_csv(DATA_PATH)
df.head()

## Data preparation

In [ ]:
# The source file uses day/month/year formatting.
df["datetime"] = pd.to_datetime(df["datetime"], format="%d/%m/%Y %H:%M")

df["hour"] = df["datetime"].dt.hour
df["date"] = df["datetime"].dt.date
df["month"] = df["datetime"].dt.month
df["weekday"] = df["datetime"].dt.day_name()

print(f"Rows: {len(df):,}")
print(f"Period: {df['datetime'].min()} to {df['datetime'].max()}")
df.dtypes

In [ ]:
df.isna().sum()

## Overall rental demand

In [ ]:
df["count"].describe().round(2)

## Demand by hour of day

In [ ]:
hourly = df.groupby("hour")["count"].mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
hourly.plot(marker="o", ax=ax)
ax.set_title("Average bike rentals by hour of day")
ax.set_xlabel("Hour")
ax.set_ylabel("Average rentals")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

hourly.sort_values(ascending=False).head()

The strongest average demand occurs around the morning and evening commuting periods, with the highest average volume at 17:00 and 18:00 and another clear peak around 08:00.

## Demand by season

In [ ]:
season_order = ["spring", "summer", "fall", "winter"]
seasonal = df.groupby("season")["count"].mean().reindex(season_order)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
seasonal.plot(kind="bar", ax=ax)
ax.set_title("Average rentals by season")
ax.set_xlabel("Season")
ax.set_ylabel("Average rentals")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

seasonal.round(2)

## Working days vs. non-working days

In [ ]:
workday = df.groupby("workingday")["count"].agg(["mean", "median", "std"]).round(2)
workday.index = workday.index.map({0: "Non-working day", 1: "Working day"})
workday

Overall daily averages are similar between working and non-working days, but this aggregate hides important differences in **when** demand occurs. On working days, commuting peaks are much more visible.

In [ ]:
hour_work = df.pivot_table(index="hour", columns="workingday", values="count", aggfunc="mean")
hour_work.columns = ["Non-working day", "Working day"]

fig, ax = plt.subplots(figsize=(9, 4.5))
hour_work.plot(ax=ax)
ax.set_title("Average hourly demand by working-day status")
ax.set_xlabel("Hour")
ax.set_ylabel("Average rentals")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Temperature, humidity and demand

In [ ]:
corr = df[["count", "temp", "humidity", "windspeed"]].corr().round(3)
corr

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].scatter(df["temp"], df["count"], alpha=0.2, s=12)
axes[0].set_title("Temperature vs. rentals")
axes[0].set_xlabel("Temperature")
axes[0].set_ylabel("Rentals")

axes[1].scatter(df["humidity"], df["count"], alpha=0.2, s=12)
axes[1].set_title("Humidity vs. rentals")
axes[1].set_xlabel("Humidity")
axes[1].set_ylabel("Rentals")

plt.tight_layout()
plt.show()

The simple correlations show a moderate positive relationship between temperature and rental volume and a negative relationship between humidity and demand. These are associations rather than causal effects; time of day, season and working-day status also influence usage.

## Temperature and humidity bands

In [ ]:
df["temp_band"] = (df["temp"] / 5).round() * 5
df["humidity_band"] = (df["humidity"] / 10).round() * 10

band_summary = (
    df.groupby(["temp_band", "humidity_band"])["count"]
      .mean()
      .reset_index(name="avg_rentals")
      .sort_values("avg_rentals", ascending=False)
)

band_summary.head(10).round(2)

## Key findings

- Bike demand has strong **time-of-day structure**, with prominent commuting peaks.
- In the included stratified sample, average demand is highest in **fall** and lowest in **spring**; the differences should be interpreted cautiously because the repository uses a compact sample.
- Working and non-working days have similar overall average demand, but very different hourly patterns.
- Higher temperature is associated with higher rental volume, while humidity is negatively associated with demand.

## Limitations

This is exploratory analysis. It does not estimate causal effects or control simultaneously for season, weather, holidays and calendar trends. A forecasting project would require a train/test design and predictive evaluation.